# Unidad 15. Excepciones y context managers

**Duración orientativa:** 7 horas · **Versión:** 2.0 · **UT2**

Los programas reales reciben datos incompletos, rutas incorrectas y valores que no se pueden convertir. Esta unidad enseña a distinguir esos casos, tratarlos sin ocultarlos y liberar recursos de forma segura.

## Resultados de aprendizaje

Al finalizar podrás leer una traza, reconocer excepciones frecuentes, elegir bloques try pequeños, usar except, else y finally, propagar errores con contexto, diseñar una política de recuperación y gestionar archivos mediante with.

## 1. Qué es una excepción

Una excepción es un objeto que comunica que una operación no puede continuar normalmente. Python interrumpe la instrucción, busca un manejador adecuado y, si no lo encuentra, muestra una traza. Una excepción no significa que el programa esté mal diseñado: puede representar una situación externa prevista, como que falte un archivo.

La última línea de la traza suele indicar el tipo y el mensaje. Las líneas anteriores muestran dónde se produjo. Lee primero el tipo, identifica la operación y reproduce el caso con el dato mínimo. No copies una traza sin explicar qué dato y qué regla han causado el fallo.

### Excepciones frecuentes

ValueError indica que el tipo puede ser correcto pero el valor no sirve, como float aplicado a una palabra. TypeError indica una operación entre tipos incompatibles. KeyError aparece al consultar una clave inexistente. IndexError aparece al pedir una posición que no existe. FileNotFoundError indica una ruta ausente y UnicodeDecodeError una codificación incorrecta.

Captura solo los tipos que tu código puede tratar. Capturar Exception de forma general puede ocultar errores de programación, como un nombre mal escrito. Durante el desarrollo es preferible que esos errores sean visibles.

In [ ]:
texto = "21.4"
try:
    temperatura = float(texto)
except ValueError:
    print("La lectura no representa un número decimal.")
else:
    print("Temperatura válida:", temperatura)


## 2. Estructura de tratamiento

Try contiene solo la operación que puede fallar. Except trata una excepción concreta. Else se ejecuta exclusivamente cuando try terminó sin excepción y es útil para separar el camino correcto. Finally se ejecuta siempre; puede servir para una limpieza que no gestione un context manager.

No coloques un bloque grande dentro de try. Si la lectura, el cálculo y la presentación están mezclados, será difícil saber cuál falló. Usa bloques pequeños y deja que cada nivel del programa gestione solo los problemas que comprende.

In [ ]:
def dividir(total: float, cantidad: int) -> float | None:
    try:
        resultado = total / cantidad
    except ZeroDivisionError:
        return None
    else:
        return resultado
    finally:
        print("División finalizada.")

print(dividir(20, 4))
print(dividir(20, 0))


## 3. Recuperar, propagar o detener

Ante una excepción existen varias políticas. Recuperar significa ofrecer una alternativa segura, por ejemplo devolver None si una media no existe. Propagar significa dejar que otro nivel trate el problema. Detener con un mensaje claro es adecuado cuando seguir produciría datos falsos.

No sustituyas un dato inválido por cero para que el programa continúe: cero podría ser un valor real. Documenta siempre qué significa None o cualquier valor alternativo. Una función de bajo nivel puede añadir contexto con raise y conservar la causa original usando from.

In [ ]:
def convertir_temperatura(texto: str) -> float:
    try:
        return float(texto)
    except ValueError as error:
        raise ValueError(f"Temperatura no válida: {texto!r}") from error

try:
    convertir_temperatura("veinte")
except ValueError as error:
    print(error)


### Mensajes y excepciones propias

Un mensaje útil identifica el contexto, el dato problemático y la expectativa. Por ejemplo, fila 4, temperatura no numérica informa más que error. No incluyas datos personales o secretos en mensajes que puedan guardarse o compartirse.

Una excepción propia solo es necesaria si el dominio debe distinguir una situación específica de otras. Para la mayoría de validaciones iniciales, ValueError y un mensaje contextual son suficientes. Crear muchas clases de excepción sin un uso claro complica el código.

## 4. Recursos y context managers

Un archivo, una conexión o un bloqueo son recursos que deben liberarse. With entra en un context manager y garantiza la salida incluso si aparece una excepción. Es la forma estándar de abrir un archivo: el bloque realiza la operación y el archivo se cierra automáticamente al terminar.

With no valida una ruta ni interpreta un formato por sí solo. Antes de abrir, decide la ruta y codificación; dentro, lee o escribe; fuera, procesa el resultado. Si puede faltar el archivo, captura FileNotFoundError en el nivel que pueda informar o decidir qué hacer.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "nota.txt"
    with ruta.open("w", encoding="utf-8") as archivo:
        archivo.write("Lectura correcta\n")
    with ruta.open("r", encoding="utf-8") as archivo:
        contenido = archivo.read()
    print(repr(contenido))


## 5. Archivos robustos

Al leer datos, separa localizar, abrir, interpretar y validar. Una ruta inexistente no es lo mismo que una fila CSV mal formada. Un CSV puede abrirse correctamente y contener una temperatura no convertible. Separar etapas permite informar con precisión y probar cada una.

Los ejemplos usan directorios temporales para no modificar materiales del curso. En una entrega real, construye una ruta relativa y evita sobrescribir recursos originales. Usa UTF-8 explícitamente y conserva número de fila, campo y representación del valor cuando una conversión falla.

## 6. Depuración de excepciones

Primero reproduce el problema con la entrada mínima. Muestra repr del texto, el tipo del valor y la ruta exacta construida con Path. Comprueba si el bloque try cubre solo la operación esperada. Después añade una prueba que garantice el comportamiento decidido para ese caso.

No uses una excepción como forma habitual de controlar el flujo si una condición sencilla expresa mejor la regla. Por ejemplo, comprueba si una lista está vacía antes de calcular una media. Las excepciones son para situaciones excepcionales o fallos de operaciones que no se pueden conocer sin intentarlas.

## Práctica guiada

1. Ejecuta los ejemplos y cambia una entrada válida por una inválida. Describe tipo, mensaje y política aplicada.
2. Mueve temporalmente una línea no relacionada dentro de try y explica por qué empeora el diagnóstico.
3. Crea una ruta temporal inexistente, captura FileNotFoundError y muestra un mensaje contextual.
4. Escribe una tabla con entrada, resultado esperado y excepción o valor alternativo.
5. Explica por qué with es preferible a abrir y cerrar manualmente el archivo.

## Ejercicios

**Ejercicio 15.01.** Crea convertir_humedad que convierta texto a entero y lance ValueError contextual si falla. Prueba texto válido e inválido.

**Ejercicio 15.02.** Crea obtener_temperatura que reciba un diccionario y un aula. Añade contexto a KeyError cuando falte el aula.

**Ejercicio 15.03.** En un directorio temporal escribe un archivo UTF-8 y léelo con with. Después prueba una ruta inexistente y gestiona FileNotFoundError.

**Ejercicio 15.04.** Compara una política que devuelve None con otra que propaga ValueError para una conversión fallida. Indica cuándo elegirías cada una.

In [21]:
# Ejercicio 15.01
def convertir_humedad(texto: str) -> int:
    """Convierte un texto en un entero"""
    try:
        return int(texto)
    except ValueError as error:
        raise ValueError(
            f"Humedad no valida: {texto}"
        ) from error

# Pruebas
print("Texto válido:", convertir_humedad("45"))

try:
    convertir_humedad("cuarenta")
except ValueError as error:
    print("Texto inválido capturado:", error)

Texto válido: 45
Texto inválido capturado: Humedad no valida: cuarenta


In [22]:
# Ejercicio 15.02
def obtener_temperatura(temperaturas: dict[str, float], aula: str) -> float:
    """Obtiene la temperatura de un aula"""
    try:
        return temperaturas[aula]
    except KeyError as error:
        raise KeyError(
            f"No existe el {aula}"
        ) from error

registro = {"Aula 101": 21.5, "Aula 102": 23.0}
print("Aula 101:", obtener_temperatura(registro, "Aula 101"))

try:
    obtener_temperatura(registro, "Aula 103")
except KeyError as error:
    print("Error al obtener la temperatura:", error)

Aula 101: 21.5
Error al obtener la temperatura: 'No existe el Aula 103'


In [17]:
# Ejercicio 15.03
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "lectura.txt"

    with ruta.open("w", encoding="utf-8") as archivo:
        archivo.write("Lectura de prueba")

    with ruta.open("r", encoding="utf-8") as archivo:
        contenido = archivo.read()
    print("Contenido leído:", repr(contenido))

ruta_inexistente = Path("directorio_falso") / "archivo.txt"
try:
    with ruta_inexistente.open("r", encoding="utf-8") as archivo:
        archivo.read()
except FileNotFoundError as error:
    print(f"{error}. No se encontró la ruta {ruta_inexistente}")

Contenido leído: 'Lectura de prueba'
[Errno 2] No such file or directory: 'directorio_falso\\archivo.txt'. No se encontró la ruta directorio_falso\archivo.txt


## Ejercicio 15.04
- Al devolver None sabes que ha fallado algo, pero no puedes saber que es precisamente. Al usar ValueError ya sabes que el contenido del dato recibido no es válido

## Proyecto 15 · Cargador robusto de lecturas

Implementa cargar_lecturas que reciba una ruta y lea un CSV con aula y temperatura. Debe usar with, propagar FileNotFoundError con contexto y lanzar ValueError indicando número de fila si una temperatura no se puede convertir. Implementa resumir que reciba solo lecturas válidas y devuelva None para una lista vacía.

Prueba en directorios temporales: archivo correcto, archivo ausente, encabezado incorrecto y temperatura inválida. Documenta qué excepción gestiona cada nivel, por qué no se devuelve una lista vacía si el archivo falta y qué información conservarías para investigar una fila errónea.

In [26]:
import csv
from pathlib import Path
from tempfile import TemporaryDirectory


def cargar_lecturas(ruta: Path) -> list[tuple[str, float]]:
    """Carga de un CSV las aulas y sus temperaturas."""
    lecturas = []

    try:
        with ruta.open(mode="r", encoding="utf-8", newline="") as archivo:
            lector = csv.reader(archivo)
            archivo_vacio = True

            for numero_fila, fila in enumerate(lector, start=1):
                archivo_vacio = False

                if numero_fila == 1:
                    if fila != ["aula", "temperatura"]:
                        raise ValueError("Encabezado incorrecto: deber ser 'aula,temperatura'")
                    continue

                if not fila:
                    continue

                if len(fila) != 2:
                    raise ValueError(f"Fila {numero_fila}: se esperaban 2 campos y hay {fila!r}")

                aula = fila[0].strip()
                temp_texto = fila[1].strip()

                try:
                    temperatura = float(temp_texto)
                except ValueError as error:
                    raise ValueError(f"Fila {numero_fila}: la temperatura {temp_texto!r} del aula {aula!r} no es válida"
                    ) from error

                lecturas.append((aula, temperatura))

            if archivo_vacio:
                raise ValueError("El archivo CSV está vacío")

    except FileNotFoundError as error:
        raise FileNotFoundError(f"No se pudo cargar el archivo. Ruta no encontrada: {ruta}"
        ) from error

    return lecturas


def resumir(lecturas: list[tuple[str, float]]) -> float | None:
    """Devuelve la temperatura media o None si no hay lecturas"""
    if not lecturas:
        return None

    total = sum(temperatura for _, temperatura in lecturas)
    return total / len(lecturas)

# PRUEBAS
with TemporaryDirectory() as carpeta_temp:
    dir_base = Path(carpeta_temp)

    # 1. Archivo correcto y función resumir
    ruta_ok = dir_base / "correctas.csv"
    with ruta_ok.open(mode="w", encoding="utf-8", newline="") as f:
        f.write("aula,temperatura\nAula 1,21.5\nAula 2,22.5\n")

    lecturas_ok = cargar_lecturas(ruta_ok)
    assert lecturas_ok == [("Aula 1", 21.5), ("Aula 2", 22.5)]
    assert resumir(lecturas_ok) == 22.0
    assert resumir([]) is None

    # 2. Archivo inexistente
    ruta_ausente = dir_base / "ausente.csv"
    try:
        cargar_lecturas(ruta_ausente)
        assert False, "Se esperaba un FileNotFoundError"
    except FileNotFoundError as error:
        assert "Ruta no encontrada" in str(error)

    # 3. Encabezado incorrecto
    ruta_encabezado = dir_base / "encabezado_incorrecto.csv"
    with ruta_encabezado.open(mode="w", encoding="utf-8", newline="") as f:
        # Se escribe un encabezado erróneo para forzar el fallo
        f.write("clase,grados\nAula 1,21.5\n")

    try:
        cargar_lecturas(ruta_encabezado)
        assert False, "Se esperaba un ValueError por el encabezado"
    except ValueError as error:
        assert "Encabezado incorrecto" in str(error)

    # 4. Temperatura inválida
    ruta_invalida = dir_base / "temp_invalida.csv"
    with ruta_invalida.open(mode="w", encoding="utf-8", newline="") as f:
        f.write("aula,temperatura\nAula 1,21.5\nAula 2,veintiuno\n")

    try:
        cargar_lecturas(ruta_invalida)
        assert False, "Se esperaba un ValueError por conversión de tipo"
    except ValueError as error:
        assert "Fila 3" in str(error)
        assert "veintiuno" in str(error)
        assert "Aula 2" in str(error)



## Autoevaluación

¿Puedes leer una traza, elegir una excepción específica, explicar try-except-else-finally, distinguir recuperación de propagación y justificar with?

- [Errores y excepciones](https://docs.python.org/3/tutorial/errors.html)
- [Context managers](https://docs.python.org/3/reference/datamodel.html#context-managers)